## Introduction
This notebook prepares data for modeling by splitting it into train, validation, and test sets and applying training-based transformations.

Close price is the target variable, and selected numerical and categorical variables are used as features. The training window length is tuned during validation.

In [18]:
import numpy as np
import pandas as pd

In [19]:
sold = pd.read_csv('clean-data/sold_clean.csv')
sold.shape

/var/folders/t2/p9112v_n469068__fty8_3nc0000gn/T/ipykernel_89841/394573382.py:1: DtypeWarning: Columns (39,40,41,42) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv('clean-data/sold_clean.csv')


(328866, 43)

## Part 1: Chronological Split
Split the dataset chronologically into training, validation, and test sets. The most recent month is reserved for testing, the second most recent month for validation, and a variable-length historical window preceding the validation period for training. The length of the training window is tuned as a hyperparameter during model validation.

In [20]:
# hyperparameter to tune
months_preceding = 12

In [21]:
close_month = pd.to_datetime(sold['CloseDate']).dt.to_period('M')
test_month = close_month.max()
validation_month = close_month.max() - 1
train_start_month = validation_month - months_preceding

In [22]:
sold_test = sold[close_month == test_month]
sold_validation = sold[close_month == validation_month]
sold_train = sold[(close_month >= train_start_month) & (close_month < validation_month)]

In [23]:
splits = {
    'Split': ['Train', 'Validation', 'Test'],
    'Size': [sold_train.shape[0], sold_validation.shape[0], sold_test.shape[0]],
    'Start Date': [sold_train['CloseDate'].min(), sold_validation['CloseDate'].min(), sold_test['CloseDate'].min()],
    'End Date': [sold_train['CloseDate'].max(), sold_validation['CloseDate'].max(), sold_test['CloseDate'].max()]
}
pd.DataFrame(splits)

,Split,Size,Start Date,End Date
0,Train,128998,2025-05-01,2026-04-30
1,Validation,11836,2026-05-01,2026-05-31
2,Test,12656,2026-06-01,2026-06-30


## Part 2: Training-Based Transformations
Learn imputation, outlier thresholds, scaling, and encoding parameters from the training data, then apply the learned transformations to the validation and test sets.

Outlier thresholds are calculated below, while all other preprocessing transformations are implemented as functions in a standalone text file for use during model development.

#### Outlier Detection
Calculate the 0.5th and 99.5th percentiles of close price from the training data. Treat any record with a close price outside these bounds as an outlier and drop these records from the training, validation, and test sets.

In [24]:
y_train = sold_train['ClosePrice']
y_validation = sold_validation['ClosePrice']
y_test = sold_test['ClosePrice']

In [25]:
lower_price = y_train.quantile(0.005)
upper_price = y_train.quantile(0.995)
print(f'Lower price threshold: ${lower_price:,.0f}')
print(f'Upper price threshold: ${upper_price:,.0f}')

Lower price threshold: $195,000
Upper price threshold: $8,500,000


In [26]:
train_price_cleaned = y_train.between(lower_price, upper_price, inclusive='both')
validation_price_cleaned = y_validation.between(lower_price, upper_price, inclusive='both')
test_price_cleaned = y_test.between(lower_price, upper_price, inclusive='both')

In [27]:
sold_train = sold_train.loc[train_price_cleaned].reset_index(drop=True)
sold_validation = sold_validation.loc[validation_price_cleaned].reset_index(drop=True)
sold_test = sold_test.loc[test_price_cleaned].reset_index(drop=True)

In [28]:
splits = {
    'Split': ['Train', 'Validation', 'Test'],
    'Size Before': splits['Size'],
    'Size After': [sold_train.shape[0], sold_validation.shape[0], sold_test.shape[0]], 
    '% Removed': [(1 - sold_train.shape[0] / splits['Size'][0]) * 100, 
                  (1 - sold_validation.shape[0] / splits['Size'][1]) * 100, 
                  (1 - sold_test.shape[0] / splits['Size'][2]) * 100]
}
pd.DataFrame(splits)

,Split,Size Before,Size After,% Removed
0,Train,128998,127721,0.989938
1,Validation,11836,11722,0.963163
2,Test,12656,12523,1.050885


Further calculate the 0.5th and 99.5th percentiles of price per square foot from the training data to filter out data entry errors and rare sales.

In [29]:
price_per_sqft_train = sold_train['ClosePrice'] / sold_train['LivingArea']
price_per_sqft_validation = sold_validation['ClosePrice'] / sold_validation['LivingArea']
price_per_sqft_test = sold_test['ClosePrice'] / sold_test['LivingArea']

In [30]:
lower_price_per_sqft = price_per_sqft_train.quantile(0.005)
upper_price_per_sqft = price_per_sqft_train.quantile(0.995)
print(f'Lower price per sqft threshold: ${lower_price_per_sqft:,.0f}')
print(f'Upper price per sqft threshold: ${upper_price_per_sqft:,.0f}')

Lower price per sqft threshold: $163
Upper price per sqft threshold: $2,081


In [31]:
train_price_per_sqft_cleaned = price_per_sqft_train.between(lower_price_per_sqft, upper_price_per_sqft, inclusive='both')
validation_price_per_sqft_cleaned = price_per_sqft_validation.between(lower_price_per_sqft, upper_price_per_sqft, inclusive='both')
test_price_per_sqft_cleaned = price_per_sqft_test.between(lower_price_per_sqft, upper_price_per_sqft, inclusive='both')

In [32]:
sold_train = sold_train.loc[train_price_per_sqft_cleaned].reset_index(drop=True)
sold_validation = sold_validation.loc[validation_price_per_sqft_cleaned].reset_index(drop=True)
sold_test = sold_test.loc[test_price_per_sqft_cleaned].reset_index(drop=True)

In [33]:
splits = {
    'Split': ['Train', 'Validation', 'Test'],
    'Size Before': splits['Size After'],
    'Size After': [sold_train.shape[0], sold_validation.shape[0], sold_test.shape[0]], 
    '% Removed': [(1 - sold_train.shape[0] / splits['Size After'][0]) * 100, 
                  (1 - sold_validation.shape[0] / splits['Size After'][1]) * 100, 
                  (1 - sold_test.shape[0] / splits['Size After'][2]) * 100]
}
pd.DataFrame(splits)

,Split,Size Before,Size After,% Removed
0,Train,127721,126443,1.000619
1,Validation,11722,11588,1.143150
2,Test,12523,12385,1.101972


## Conclusion
The dataset has been split into training, validation, and test sets, with each set transformed using preprocessing parameters learned exclusively from the training data to ensure reliable model evaluation. 

Export the transformed datasets for subsequent modeling.

In [43]:
sold_train.to_csv('clean-data/sold_train.csv', index=False)
sold_validation.to_csv('clean-data/sold_validation.csv', index=False)
sold_test.to_csv('clean-data/sold_test.csv', index=False)